In [1]:
import lynse
lynse.__version__

'0.2.0'

In [2]:
import lynse
client = lynse.VectorDBClient('test/')
my_db = client.create_database("my_vec_db", drop_if_exists=True)
# create or truncate a collection
collection = my_db.require_collection("test_add_vectors", dim=128, drop_if_exists=True,
                                      cache_chunks=-1, cache_query=False, chunk_size=10_0000)

In [1]:
import lynse
import numpy as np
import pandas as pd
from tqdm import trange

client = lynse.VectorDBClient('test/')
my_db = client.create_database("my_vec_db", drop_if_exists=True)
# create or truncate a collection
collection = my_db.require_collection("test_add_vectors", dim=128, drop_if_exists=True,
                                      cache_chunks=0, cache_query=False, chunk_size=10_0000)

d = 128
n = 100_0000
current_rows = collection.shape[0]

with collection.insert_session() as session:
    for i in trange(current_rows, current_rows + n):
        if i == 0:
            query = np.random.random(d)
            vec = query
        else:
            vec = np.random.random(d)
    
        session.add_item(vec, field={"id": i, "test": f"test_{i // 1000}", "order": i})

collection.build_index(index_mode="IVF-IP")
collection.search(query, k=10)

100%|████████████| 1000000/1000000 [00:07<00:00, 141314.17it/s]
2025-07-11 13:36:20.150749 | INFO     | Saving data...
2025-07-11 13:36:20.150749 | INFO     | Writing chunk to storage...
Data persisting: 100%|██████| 10/10 [00:03<00:00,  3.00chunk/s]
2025-07-11 13:36:23.660135 | INFO     | Writing chunk to storage done.
2025-07-11 13:36:23.660135 | INFO     | Pre-building the index...
2025-07-11 13:36:23.660135 | INFO     | Building an index using the `flat-ip` index mode...
2025-07-11 13:36:24.632962 | INFO     | Index built.
2025-07-11 13:36:24.632962 | INFO     | Building an index using the `ivf-ip` index mode...
2025-07-11 13:36:32.068153 | INFO     | Index built.


SearchResult(dim=128, k=10, index_mode='ivf-ip', res_num=10)

In [5]:
collection.search(query, k=10).to_df()

,id,distance,fields
0,0,45.017700,"{'id': 0, 'order': 0, 'test': 'test_0'}"
1,978229,42.629227,"{'id': 978229, 'order': 978229, 'test': 'test_..."
2,690806,42.402061,"{'id': 690806, 'order': 690806, 'test': 'test_..."
3,218319,42.368549,"{'id': 218319, 'order': 218319, 'test': 'test_..."
4,329525,42.191040,"{'id': 329525, 'order': 329525, 'test': 'test_..."
5,465712,42.086060,"{'id': 465712, 'order': 465712, 'test': 'test_..."
6,12643,41.995174,"{'id': 12643, 'order': 12643, 'test': 'test_12'}"
7,533025,41.965546,"{'id': 533025, 'order': 533025, 'test': 'test_..."
8,359089,41.939491,"{'id': 359089, 'order': 359089, 'test': 'test_..."
9,717872,41.704453,"{'id': 717872, 'order': 717872, 'test': 'test_..."


In [15]:
%timeit collection.search(query, k=10)

22.7 ms ± 213 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
collection.index_mode

'flat-ip'

In [17]:
collection.search(query, k=10).to_df()

,index,distance,id,order,test
0,0,42.925705,0,0,test_0
1,118154,39.495529,118154,118154,test_118
2,238824,39.317459,238824,238824,test_238
3,115986,39.297729,115986,115986,test_115
4,199681,39.164116,199681,199681,test_199
5,996289,39.163658,996289,996289,test_996
6,949416,39.096947,949416,949416,test_949
7,511660,39.014694,511660,511660,test_511
8,215324,38.984444,215324,215324,test_215
9,937318,38.971004,937318,937318,test_937


In [19]:
collection.head(10).to_df()

,vectors,id,order,test
0,"[0.8166013956069946, 0.1753922551870346, 0.420...",0,0,test_0
1,"[0.6288195848464966, 0.5755744576454163, 0.711...",1,1,test_0
2,"[0.7420116662979126, 0.7844901084899902, 0.295...",2,2,test_0
3,"[0.37366822361946106, 0.3597498834133148, 0.28...",3,3,test_0
4,"[0.04122208058834076, 0.8603161573410034, 0.72...",4,4,test_0
5,"[0.07475878298282623, 0.7868481278419495, 0.84...",5,5,test_0
6,"[0.06956613063812256, 0.6551972031593323, 0.27...",6,6,test_0
7,"[0.23337776958942413, 0.5618239045143127, 0.03...",7,7,test_0
8,"[0.2756570279598236, 0.7753728032112122, 0.776...",8,8,test_0
9,"[0.7015456557273865, 0.18231110274791718, 0.76...",9,9,test_0


In [1]:
import lynse
import numpy as np
import pandas as pd
from tqdm import trange


client = lynse.VectorDBClient('test/')
my_db = client.create_database("my_vec_db", drop_if_exists=False)
# create or truncate a collection
collection = my_db.require_collection("test_add_vectors", dim=128, drop_if_exists=True,
                                      cache_chunks=-1, cache_query=False, chunk_size=10_0000)


d = 128
n = 100_0000
current_rows = collection.shape[0]

a = {"vectors":[], "id":[], "test":[], "order":[]}
# with collection.insert_session() as session:
for i in trange(current_rows, current_rows + n):
    if i == 0:
        query = np.random.random(d)
        vec = query
    else:
        vec = np.random.random(d)

    a["vectors"].append(vec)
    a["id"].append(i)
    a["test"].append(f"test_{i // 1000}")
    a["order"].append(i)
    # session.add_item(vec, field={"id": i, "test": f"test_{i // 1000}", "order": i})

pd_a = pd.DataFrame(a)
with collection.insert_session() as session:
    collection.from_pandas(pd_a)

collection.search(query, k=10)


2025-07-08 17:41:52 - LynseDB - INFO - Pre-building the index...
2025-07-08 17:41:53 - LynseDB - INFO - Existing index loaded and synchronized.
100%|██████████| 1000000/1000000 [00:01<00:00, 649695.77it/s]ectors' already exists. Dropped.

2025-07-08 17:42:00 - LynseDB - INFO - Saving data...
2025-07-08 17:42:00 - LynseDB - INFO - Writing chunk to storage...
Data persisting:  10%|█         | 1/10 [00:04<00:39,  4.40s/chunk]

2025-07-08 17:42:04 - LynseDB - INFO - Writing chunk to storage done.
2025-07-08 17:42:04 - LynseDB - INFO - Pre-building the index...
2025-07-08 17:42:04 - LynseDB - INFO - Building an index using the `flat-ip` index mode...


{
    "indices": [
        0,
        242737,
        100660,
        957428,
        512434,
        959904,
        584650,
        972331,
        328191,
        950193
    ],
    "distances": [
        42.2326545715332,
        41.20012664794922,
        40.55289077758789,
        39.75053405761719,
        39.66271209716797,
        39.55134582519531,
        39.549285888671875,
        39.5372428894043,
        39.507286071777344,
        39.432090759277344
    ],
    "metadata": [
        {
            "id": 0,
            "order": 0,
            "test": "test_0"
        },
        {
            "id": 242737,
            "order": 242737,
            "test": "test_242"
        },
        {
            "id": 100660,
            "order": 100660,
            "test": "test_100"
        },
        {
            "id": 957428,
            "order": 957428,
            "test": "test_957"
        },
        {
            "id": 512434,
            "order": 512434,
            "test": "test

In [2]:
collection.shape

(1000000, 128)

In [5]:
collection.head(20)

                                              vectors  id  order    test
0   [0.3253602683544159, 0.018720095977187157, 0.8...   0      0  test_0
1   [0.7647805213928223, 0.8397876024246216, 0.873...   1      1  test_0
2   [0.5157454609870911, 0.5235193967819214, 0.061...   2      2  test_0
3   [0.19984233379364014, 0.3990980088710785, 0.48...   3      3  test_0
4   [0.4452367424964905, 0.26839104294776917, 0.79...   4      4  test_0
5   [0.09092839807271957, 0.6834369897842407, 0.79...   5      5  test_0
6   [0.5811700224876404, 0.29839837551116943, 0.92...   6      6  test_0
7   [0.6052560806274414, 0.7409024238586426, 0.936...   7      7  test_0
8   [0.956204891204834, 0.4049047827720642, 0.0025...   8      8  test_0
9   [0.9763140082359314, 0.9951077103614807, 0.435...   9      9  test_0
10  [0.7666349411010742, 0.782912015914917, 0.9002...  10     10  test_0
11  [0.9336947202682495, 0.16778342425823212, 0.32...  11     11  test_0
12  [0.01281452551484108, 0.36341676115989685, 0.6.

In [4]:
collection._matrix_serializer.storage_worker.id_mapper.id_map

{'chunk_0': (0, 99999),
 'chunk_1': (99999, 199999),
 'chunk_2': (199999, 299999),
 'chunk_3': (299999, 399999),
 'chunk_4': (399999, 499999),
 'chunk_5': (499999, 599999),
 'chunk_6': (599999, 699999),
 'chunk_7': (699999, 799999),
 'chunk_8': (799999, 899999),
 'chunk_9': (899999, 999999)}

In [7]:
collection.build_index("IVF-Cos")

TypeError: IVFCreator.build_index() got an unexpected keyword argument 'nlist'

In [46]:
collection.shape

(1000000, 128)

In [53]:
collection.tail(10)

                                             vectors      id   order      test
0  [0.47002479434013367, 0.9706178307533264, 0.15...  999990  999990  test_999
1  [0.8247489333152771, 0.7167617082595825, 0.785...  999991  999991  test_999
2  [0.9799009561538696, 0.10836271941661835, 0.90...  999992  999992  test_999
3  [0.606682538986206, 0.32562056183815, 0.132553...  999993  999993  test_999
4  [0.08036496490240097, 0.4096981883049011, 0.72...  999994  999994  test_999
5  [0.9467604756355286, 0.8732396960258484, 0.024...  999995  999995  test_999
6  [0.06393465399742126, 0.43599992990493774, 0.1...  999996  999996  test_999
7  [0.4908878207206726, 0.8869771361351013, 0.823...  999997  999997  test_999
8  [0.6467986702919006, 0.019344080239534378, 0.4...  999998  999998  test_999
9  [0.5419594645500183, 0.7359902262687683, 0.872...  999999  999999  test_999

In [55]:
import pandas as pd

collection.search(query, k=10).to_tuple()

(array([     0, 624647, 679947, 188276, 838144, 768147, 982640, 254394,
        653347, 405813]),
 array([43.063374, 40.990093, 40.439728, 40.068626, 40.002106, 39.943314,
        39.912693, 39.899673, 39.82421 , 39.804726], dtype=float32),
 [{'id': 188275, 'order': 188275, 'test': 'test_188'},
  {'id': 254393, 'order': 254393, 'test': 'test_254'},
  {'id': 405812, 'order': 405812, 'test': 'test_405'},
  {'id': 624646, 'order': 624646, 'test': 'test_624'},
  {'id': 653346, 'order': 653346, 'test': 'test_653'},
  {'id': 679946, 'order': 679946, 'test': 'test_679'},
  {'id': 768146, 'order': 768146, 'test': 'test_768'},
  {'id': 838143, 'order': 838143, 'test': 'test_838'},
  {'id': 982639, 'order': 982639, 'test': 'test_982'}])

In [9]:
%timeit collection.search(query, k=100)

77.7 ms ± 1.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [10]:
%timeit tbl.search(query).limit(100).to_pandas()

49.9 ms ± 557 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [13]:
collection.build_index("IVF-IP")

In [14]:
%timeit collection.search(query, k=10, nprobe=20)

21.9 ms ± 300 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [16]:
collection.remove_index()


2024-10-12 20:33:27 - LynseDB - INFO - Index removed.
2024-10-12 20:33:27 - LynseDB - INFO - Fallback to default index mode: `Flat-IP`.
2024-10-12 20:33:27 - LynseDB - INFO - Building an index using the `Flat-IP` index mode...


In [5]:
query = collection.head(1)[1]
query

array([[0.5119091 , 0.1945767 , 0.01754114, ..., 0.28386247, 0.96214914,
        0.82311755]], dtype=float32)

In [18]:
%timeit collection.search(query, k=6, search_filter=":id: in [0, 1, 2, 3] and :test: == 'test_0'")

1.7 ms ± 19.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [7]:
%%timeit
collection.search(query, k=6, search_filter=":test: == 'test_0'")

217 ms ± 12.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
%%timeit
collection.search(query, k=6, search_filter=":test: == 'test_0'")

215 ms ± 2.81 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
collection.shape

(1000000, 1024)

In [10]:
collection.remove_field_index(":test:")

In [13]:
collection.search(query, k=6, search_filter=":id: in [0, 1, 2, 3] and :test: == 'test_0'")

(array([0, 1, 2, 3]),
 array([328.2731 , 248.43213, 246.88026, 243.90518], dtype=float32),
 None)

In [14]:
collection.search(query, k=6, search_filter=":test: == 'test_0'")

(array([  0, 327, 354, 277, 207,  83]),
 array([328.2731 , 266.4355 , 262.61432, 262.00818, 262.00226, 261.48373],
       dtype=float32),
 None)

In [16]:
collection.search(query, k=6, search_filter=":test: == 'test_999'")

(array([999735, 999309, 999102, 999555, 999646, 999359]),
 array([269.6776 , 265.1906 , 264.6823 , 263.21765, 262.9951 , 261.0382 ],
       dtype=float32),
 None)

In [19]:
from lynse.field_models import *

collection.search(
            query, k=6, search_filter=Filter(
                must=[FieldCondition(key=":id:", matcher=MatchID([0, 1, 3, 2]))],
                any=[FieldCondition(key=":test:", matcher=MatchField('test_0'))],
            )
)

(array([0, 1, 2, 3]),
 array([328.2731 , 248.43213, 246.88026, 243.90518], dtype=float32),
 None)

In [20]:
%%timeit 

collection.search(
            query, k=6, search_filter=Filter(
                must=[FieldCondition(key=":id:", matcher=MatchID([0, 1, 2, 3]))],
                any=[FieldCondition(key=":test:", matcher=MatchField('test_0'))],
            )
)


1.72 ms ± 30.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
